# 00 · Cuaderno maestro / *Master notebook*
**GUAIRA-SAR** · SRL *2026 Venezuela, Colombia and Panama Earthquakes*

🇪🇸 **Un solo cuaderno** que llama a todo el flujo (inventario → descargas → InSAR → daño → figuras) usando el código de GitHub (`src/`). Cada paso se ejecuta **una vez**: queda registrado en Drive (`05_documentacion/pasos_ejecutados.csv`) y se salta solo la próxima vez. Los pasos ya hechos el 19-nov figuran como ✅ *previo*. Para repetir uno a propósito: `P("id", forzar=True)`.
🇬🇧 **One notebook** that calls the whole workflow using the GitHub code. Each step runs **once**; it is logged on Drive and skipped next time. Steps done on 19 Nov show as ✅ *previo*. To repeat on purpose: `P("id", forzar=True)`.

⚠️ 🇪🇸 **HyP3:** el envío identifica trabajos por *par de escenas + resolución* (no por nombre) y **solo envía lo que no existe**. Hoy solo falta 033A a 40 m.
⚠️ 🇬🇧 **HyP3:** submissions are matched by *scene pair + resolution*; only what does not exist is sent. Today only 033A at 40 m is missing.
🔒 🇪🇸 Credenciales en **Secretos de Colab** (🔑 en la barra lateral): `EARTHDATA_USER` y `EARTHDATA_PASS`. No se escriben en el cuaderno. 🇬🇧 Credentials in **Colab Secrets**; never typed into the notebook.
📝 Contexto completo / full context: `docs/HANDOFF.md`.

## 0 · Preparar entorno / *Setup*

In [22]:
NOTEBOOK = "00_Maestro"
!pip install -q asf_search hyp3_sdk rasterio geopandas shapely contextily pyarrow duckdb boto3 pystac-client requests h5py pyproj scipy huggingface_hub cmcrameri
import os, subprocess, datetime
REPO = "/content/SAR_multifrequency"
CUADERNO_VERSION = "2026-11-26.6"      # versión de ESTE cuaderno / version of THIS notebook
def _git(*a):
    return subprocess.run(["git", "-C", REPO, *a], capture_output=True, text=True).stdout.strip()
if os.path.isdir(REPO):
    _antes = _git("rev-parse", "HEAD")
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    _log = _git("log", "--oneline", "--no-merges", f"{_antes}..HEAD") if _antes else ""
    print("Cambios nuevos en GitHub desde la última vez / new GitHub changes since last time:")
    print(_log[:1800] if _log else "   (ninguno / none)")
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/jf-floresriera/SAR_multifrequency", REPO], check=True)
_vf = f"{REPO}/notebooks/VERSION_CUADERNO.txt"
_v = open(_vf).read().strip() if os.path.exists(_vf) else ""
if _v and _v != CUADERNO_VERSION:
    print()
    print(f"⚠️ ESTE cuaderno es la versión {CUADERNO_VERSION} y en GitHub hay la {_v}: puede faltar alguna celda nueva.")
    print("   Ábralo de nuevo desde / reopen from: https://colab.research.google.com/github/jf-floresriera/SAR_multifrequency/blob/main/notebooks/00_Maestro.ipynb")
    print("   Mientras tanto, cualquier paso se corre en una celda nueva con: P(\"id_del_paso\", forzar=True)  (ids al final / ids at the end)")
SRC = f"{REPO}/src"
VERSION = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

from google.colab import drive
drive.mount("/content/drive")
RAIZ = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR"
TRABAJO = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/inventario_2026"
for d in (TRABAJO, f"{RAIZ}/05_documentacion"):
    os.makedirs(d, exist_ok=True)
os.chdir(TRABAJO)
with open(f"{RAIZ}/05_documentacion/versiones_codigo.csv", "a") as f:   # trazabilidad / traceability
    f.write(f"{datetime.datetime.now():%Y-%m-%d %H:%M},{NOTEBOOK},{VERSION}\n")
import sys, importlib
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import comun, maestro; importlib.reload(comun); importlib.reload(maestro)
LOGS = f"{RAIZ}/05_documentacion/logs"
def correr(script):
    """Ejecuta src/<script> y guarda todo lo impreso en 05_documentacion/logs / runs and logs everything."""
    return comun.correr(script, LOGS, SRC, NOTEBOOK, VERSION)
def P(pid, **kw):
    """Ejecuta un paso del flujo salvo que ya esté hecho / run a workflow step unless already done."""
    return maestro.paso(pid, correr, RAIZ, VERSION, **kw)
print(f"Código / code: {VERSION} · Carpeta / folder: {os.getcwd()}")
print("Pasos disponibles / available steps:", maestro.IDS)

Cambios nuevos en GitHub desde la última vez / new GitHub changes since last time:
bec4cd1 Figura de comparacion de todos los modelos (familia x ASC/DESC), banderas Applied en la auditoria, plan principal/suplemento
35b08ee Mosaico unificado por (familia de sensor, direccion): NISAR, Sentinel-1, ALOS-2 y NISAR de UCSD; sin mezclar sensores
33373c9 Mosaico NISAR (solape/campo lejano) y fix: sensor mal etiquetado entre archivos UCSD en insar_metricas
16e532a NISAR: un archivo por frame (antes el segundo frame sobrescribia al primero); comparacion con UCSD une frames; HANDOFF
7726d38 Maestro: 'Ejecutar todo' seguro (sin forzar en pasos pesados; qc/auditoria/insar_metricas siempre re-ejecutables); version cuaderno .4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Código / code: bec4cd1 · Carpeta / folder: /content/drive/My Drive/InSAR_LaGuaira_Resultados/inventario_2026
Pasos disponibles / available steps: ['

## 1 · Credenciales NASA Earthdata (Secretos de Colab) / *Credentials (Colab Secrets)*

In [23]:
import asf_search as asf
try:
    from google.colab import userdata
    usuario, clave = userdata.get("EARTHDATA_USER"), userdata.get("EARTHDATA_PASS")
except Exception:
    import getpass
    print("Sin Secretos de Colab: se pedirán aquí (la clave no se muestra). / No Colab Secrets: prompting here.")
    usuario = input("Usuario Earthdata / username: "); clave = getpass.getpass("Contraseña / password: ")
session = asf.ASFSession().auth_with_creds(usuario, clave)
print("Autenticación correcta / authenticated")

Autenticación correcta / authenticated


## 2 · Panel de estado / *Status board*
✅ hecho · ♻️ se puede correr siempre · ⏳ pendiente

In [24]:
import pandas as pd; pd.set_option("display.max_colwidth", 90, "display.width", 200)
maestro.estado(RAIZ)

,paso / step,fase / phase,estado / status,descripción (ES),description (EN),detalle / detail
0,inv_umbra,A Inventario,✅,"Umbra X: 26 adquisiciones, STAC, huellas","Umbra X: 26 collects, STAC, footprints",previo / earlier · 2026-11-19 · carpeta/folder inventario_2026/umbra_inventario
1,inv_nisar,A Inventario,✅,NISAR L: GUNW cosísmicos,NISAR L: coseismic GUNW,previo / earlier · 2026-11-19 · inventario_2026/nisar_filtrado/nisar_productos.csv
2,inv_s1,A Inventario,✅,Sentinel-1 C: pares cosísmicos (solo búsqueda),Sentinel-1 C: coseismic pairs (search only),"previo / earlier · 2026-11-19 · trabajos/jobs HyP3 025D, Path33"
3,inv_alos2,A Inventario,✅,ALOS-2 L: búsqueda en ASF,ALOS-2 L: ASF search,2026-11-26 19:40 · alos2_buscar.py
4,inv_tabla,A Inventario,✅,Tabla maestra y línea de tiempo (suplemento),Master table and timeline (supplement),previo / earlier · 2026-11-19 · cuaderno/notebook Inventario_Multisensor (timeline)
5,mapas_base,A Inventario,✅,Mapas base y figura de ubicación (Fig. 1),Base maps and location figure (Fig. 1),2026-11-26 22:13 · mapas_base.py md5:d9091a94
6,hyp3_estado,B Descargas,✅,Estado de HyP3 y descarga de lo terminado,HyP3 status and download of finished jobs,2026-11-26 22:13 · hyp3_estado.py md5:6847f2f9
7,hyp3_envio,B Descargas,⏳,Enviar a HyP3 (SOLO lo que no existe; gasta créditos),Submit to HyP3 (ONLY what does not exist; uses credits),pendiente / pending
8,descargas,B Descargas,✅,"NISAR, Umbra GEC y Sentinel-1 a Drive","NISAR, Umbra GEC and Sentinel-1 to Drive",2026-11-26 22:18 · descargas.py md5:9eb5c1c2
9,auxiliares,C InSAR,✅,Réplicas USGS y WorldCover,USGS aftershocks and WorldCover,2026-11-26 20:33 · auxiliares.py


## A · Inventario multisensor / *Multi-sensor inventory*
🇪🇸 Umbra (X), NISAR y ALOS-2 (L), Sentinel-1 (C), Planet. Estas figuras y tablas van al **material suplementario** (S1–S3); una versión sintética alimenta la Fig. 1. 🇬🇧 These go to the **electronic supplement**.

In [ ]:
P("inv_umbra")
P("inv_nisar")

In [ ]:
P("inv_s1", variables={"ENVIAR_HYP3": False})   # solo busca / search only

In [ ]:
P("inv_alos2")   # ALOS-2: si ASF no lo tiene, ver docs/HANDOFF.md §5 (fecha de corte 5-dic) / see HANDOFF §5

In [ ]:
P("inv_tabla")
P("mapas_base")   # se repite solo si cambia su código / re-runs by itself when its code changes

## B · HyP3 y descargas / *HyP3 and downloads*
🇪🇸 `hyp3_estado` descarga lo terminado a Drive (HyP3 borra a los ~14 días). `hyp3_envio` está **bloqueado** hasta poner `confirmar=True`: enviará SOLO lo que no exista (hoy: 033A a 40 m, 1 trabajo). 🇬🇧 `hyp3_envio` is locked until `confirmar=True`.

In [ ]:
P("hyp3_estado", variables={"DESCARGAR": True})

In [ ]:
P("hyp3_envio", confirmar=False, variables={"ENVIAR_HYP3": True})   # ← cambiar a confirmar=True para enviar / set True to submit

In [ ]:
P("descargas", variables={"ESPERAR_HYP3": False})   # NISAR (ya en Drive se omite), Umbra GEC, Sentinel-1

## C · InSAR banda C y L / *C- and L-band InSAR*
🇪🇸 Escriba las coordenadas de CCS1 (ficha NGL / IGVSB) para validar con GNSS. 🇬🇧 Enter CCS1 coordinates to validate.

In [25]:
P("auxiliares")
P("nisar_tif")
P("s1_tif")

⏭️  auxiliares: ya hecho / already done. Use forzar=True para repetir / to repeat.
🔄 nisar_tif: el código cambió desde la última vez: se repite solo / code changed since the last run: re-running

== NISAR_T162A_F006_20260613_20260625
{
 "producto": "NISAR_L2_PR_GUNW_022_162_A_006_023_4000_SH_20260613T100623_20260613T100657_20260625T100622_20260625T100656_P05023_N_F_J_001",
 "nombre_corto": "NISAR_T162A_F006_20260613_20260625",
 "track": "162A",
 "frame": "006",
 "epsg": 32619,
 "lambda_m": 0.24196,
 "fecha_ref": "20260613",
 "fecha_sec": "20260625",
 "tiene_ionosfera": true,
 "tiene_vectores_los": true,
 "los_p2_m": -0.568706214427948,
 "los_p98_m": 0.0,
 "coh_media": 0.40303364396095276
}

== NISAR_T162A_F007_20260613_20260625
{
 "producto": "NISAR_L2_PR_GUNW_022_162_A_007_023_4000_SH_20260613T100656_20260613T100731_20260625T100655_20260625T100730_P05023_N_F_J_001",
 "nombre_corto": "NISAR_T162A_F007_20260613_20260625",
 "track": "162A",
 "frame": "007",
 "epsg": 32619,
 "lambda_m": 0

In [26]:
P("descomp", variables={"CCS1_LON": None, "CCS1_LAT": None})   # ej./e.g. -66.88, 10.49 (verificar)

⏭️  descomp: ya hecho / already done. Use forzar=True para repetir / to repeat.


### C0 · Auditoría de correcciones / Corrections audit
🇪🇸 LEE cada GUNW de NISAR y cada ZIP de HyP3 y lista qué correcciones trae (ionosfera, troposfera, mareas, DEM…). Ver docs/CORRECCIONES_INSAR.md.
🇬🇧 READS every NISAR GUNW and HyP3 ZIP and lists which corrections it carries. See docs/CORRECCIONES_INSAR.md.

In [27]:
P("auditoria")

         sensor                                  capa_o_correccion                            presente                                                                                    ejemplo
     NISAR GUNW        BANDERA azimuthIonosphericCorrectionApplied                                True                                                                                           
     NISAR GUNW             BANDERA commonBandAzimuthFilterApplied                               False                                                                                           
     NISAR GUNW               BANDERA commonBandRangeFilterApplied                               False                                                                                           
     NISAR GUNW               BANDERA ellipsoidalFlatteningApplied                                True                                                                                           
     NISAR GUNW   BANDERA hydr

True

### C0b · Control de calidad de cada interferograma / Per-interferogram QC
🇪🇸 ¿La deformación realmente se ve? Muestra LOS con estiramiento robusto, **franjas** (fase re-envuelta), coherencia, histograma, y escribe un diagnóstico por producto. Ejecutar ANTES de interpretar mapas.
🇬🇧 Is the deformation really visible? Robust-stretch LOS, **fringes** (re-wrapped phase), coherence, histogram and a per-product diagnosis. Run BEFORE interpreting maps.

In [28]:
P("qc")

Máscara de tierra desde la capa «paises_sudamerica_ne50»: 58 % de la ventana es tierra
   barrido de coherencia / sweep: {'cobertura_coh0.1_pct': 90.1, 'ruido_coh0.1_mm': 104.2, 'cobertura_coh0.2_pct': 64.0, 'ruido_coh0.2_mm': 105.3, 'cobertura_coh0.3_pct': 27.4, 'ruido_coh0.3_mm': 115.2, 'cobertura_coh0.4_pct': 13.7, 'ruido_coh0.4_mm': 145.1}
S1_20260618_20260625_0B32: p2/p98 = -119/+188 mm · franjas ≈ 11.1 · SNR ≈ 1.3 · la señal NO destaca del ruido (rango p2–p98 < 5× ruido; ojo: una señal muy localizada puede no verse en percentiles): revisar par/desenvolvimiento/máscara; 42 % de la huella es MAR (incoherente): la cobertura global subestima la de tierra (38 % de la tierra supera coh ≥ 0.3); solo 27 % de los píxeles supera coh ≥ 0.3
   barrido de coherencia / sweep: {'cobertura_coh0.1_pct': 65.6, 'ruido_coh0.1_mm': 35.0, 'cobertura_coh0.2_pct': 45.1, 'ruido_coh0.2_mm': 34.4, 'cobertura_coh0.3_pct': 17.3, 'ruido_coh0.3_mm': 34.4, 'cobertura_coh0.4_pct': 7.8, 'ruido_coh0.4_mm': 37.0}
S

True

### C1b · ALOS-2 y NISAR procesados por UCSD / UCSD-processed ALOS-2 and NISAR
🇪🇸 Descarga 7 archivos de topex.ucsd.edu (~445 MB cada uno, ~3 GB en total; se guardan en 01_datos_crudos/ucsd_los) y los convierte a GeoTIFF sobre nuestra malla. Decide el signo comparando con NUESTRO NISAR. **No redistribuir** (licencia no declarada): citar a UCSD/Scripps. Tarda unos minutos.
🇬🇧 Downloads 7 files from topex.ucsd.edu (~445 MB each, ~3 GB total) and converts them to GeoTIFF on our grid; sign decided by comparison with OUR NISAR. **Do not redistribute** (licence not stated): cite UCSD/Scripps.

In [29]:
P("ucsd_los")   # prueba rápida / quick test: variables={"SOLO_TRACKS": ["NSR_T162A"]}

🔄 ucsd_los: el código cambió desde la última vez: se repite solo / code changed since the last run: re-running
✓ UCSD_NISAR_T162A_20260613_20260625: 23,651,128 filas → 2,866,270 celdas de 0.0008° · LOS p2/p98 = -37.4/13.9 cm · corr. con el nuestro = 0.429
✓ UCSD_NISAR_T061A_20260618_20260630: 9,702,429 filas → 435,386 celdas de 0.0008° · LOS p2/p98 = -64.7/-1.8 cm · corr. con el nuestro = 0.944
✓ UCSD_NISAR_T054D_20260617_20260629: 18,229,506 filas → 2,212,732 celdas de 0.0008° · LOS p2/p98 = -19.2/29.8 cm · corr. con el nuestro = 0.643
✓ UCSD_NISAR_T126D_20260622_20260704: 12,121,752 filas → 744,945 celdas de 0.0008° · LOS p2/p98 = -7.1/43.6 cm · corr. con el nuestro = -0.032
   descargando / downloading ALOS2_T040A (~445 MB)…
✓ UCSD_ALOS2_T040A_20250114_20260630: 1,590,686 filas → 1,263,421 celdas de 0.0008° · LOS p2/p98 = -9.9/48.0 cm · corr. con el nuestro = nan
   descargando / downloading ALOS2_T134D (~445 MB)…
✓ UCSD_ALOS2_T134D_20260610_20260708: 2,673,288 filas → 2,020,620 cel

True

### C1c · Mosaico de frames/tracks NISAR vecinos / Mosaic of adjacent NISAR frames/tracks
🇪🇸 Une los frames y tracks de la misma dirección (ASC: T162A+T061A; DESC: T054D+T126D, como hace UCSD), **igualando el desfase en el solape**, y mide la costura (RMS). Es para VER y comparar; Este/Vertical sigue usando los productos separados.
🇬🇧 Merges same-direction frames/tracks (ASC: T162A+T061A; DESC: T054D+T126D, as UCSD does), **matching the offset in the overlap**, and reports the seam (RMS). For viewing/comparison; East/Up still uses the separate products.

In [30]:
P("mosaico")


== ALOS-2 (UCSD) ASC: ['UCSD_ALOS2_T040A_20250114_20260630']
   (un solo producto: no hay vecino que unir / single product: nothing to merge)

== ALOS-2 (UCSD) DESC: ['UCSD_ALOS2_T134D_20260610_20260708', 'UCSD_ALOS2_T135D_20260615_20260629']
   UCSD_ALOS2_T135D_20260615_20260629: solape 1,215,710 px · desfase -36 mm (solape) · RMS solape 34 mm

== NISAR ASC: ['NISAR_T162A_F007_20260613_20260625', 'NISAR_T061A_F007_20260618_20260630', 'NISAR_T162A_F006_20260613_20260625']
   NISAR_T061A_F007_20260618_20260630: solape 116,541 px · desfase -516 mm (solape) · RMS solape 147 mm
   NISAR_T162A_F006_20260613_20260625: solape 61,361 px · desfase +125 mm (solape) · RMS solape 171 mm

== NISAR DESC: ['NISAR_T126D_F083_20260622_20260704', 'NISAR_T054D_F083_20260617_20260629', 'NISAR_T126D_F084_20260622_20260704']
   NISAR_T054D_F083_20260617_20260629: solape 103,160 px · desfase +157 mm (solape) · RMS solape 44 mm
   NISAR_T126D_F084_20260622_20260704: solape 76,536 px · desfase -620 mm (solape

True

### C1d · Figura de comparación de todos los modelos / Comparison figure of all models
🇪🇸 Filas = familia de sensor (Sentinel-1, NISAR, NISAR-UCSD, ALOS-2-UCSD), columnas = ASC | DESC; misma ventana y escala. **Figura principal**; los productos individuales y el QC van al suplemento.
🇬🇧 Rows = sensor family, columns = ASC | DESC; same window and scale. **Main figure**; individual products and QC go to the supplement.

In [31]:
P("comparacion")   # opcional / optional: variables={"LIM_MM": 400}

TypeError: 'DataFrame' object is not callable

In [32]:
# ▶ Actualizar sin reabrir el cuaderno / Refresh without reopening the notebook
import subprocess, importlib, sys
REPO = "/content/SAR_multifrequency"
r = subprocess.run(["git", "-C", REPO, "pull", "-q"], capture_output=True, text=True)
print(r.stdout or r.stderr or "código al día / code up to date")
VERSION = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")
import comun, maestro, estilo_figuras
for m in (comun, maestro, estilo_figuras):      # recargar: si no, Python sigue usando la versión vieja en memoria
    importlib.reload(m)
def correr(script):
    return comun.correr(script, LOGS, SRC, NOTEBOOK, VERSION)
def P(pid, **kw):
    return maestro.paso(pid, correr, RAIZ, VERSION, **kw)
print(f"Código / code: {VERSION}")
print("Pasos disponibles / available steps:", maestro.IDS)

código al día / code up to date
Código / code: e164fbf
Pasos disponibles / available steps: ['inv_umbra', 'inv_nisar', 'inv_s1', 'inv_alos2', 'inv_tabla', 'mapas_base', 'hyp3_estado', 'hyp3_envio', 'descargas', 'auxiliares', 'nisar_tif', 's1_tif', 'descomp', 'ucsd_los', 'auditoria', 'qc', 'mosaico', 'comparacion', 'insar_metricas', 'danio_ref', 'vantor', 'danio_sitios', 'exportar']


In [33]:
P("nisar_tif")        # frames corregidos (tarda unos minutos)
P("mosaico")
P("comparacion")
P("insar_metricas")

⏭️  nisar_tif: ya hecho / already done. Use forzar=True para repetir / to repeat.

== ALOS-2 (UCSD) ASC: ['UCSD_ALOS2_T040A_20250114_20260630']
   (un solo producto: no hay vecino que unir / single product: nothing to merge)

== ALOS-2 (UCSD) DESC: ['UCSD_ALOS2_T134D_20260610_20260708', 'UCSD_ALOS2_T135D_20260615_20260629']
   UCSD_ALOS2_T135D_20260615_20260629: solape 1,215,710 px · desfase -36 mm (solape) · RMS solape 34 mm

== NISAR ASC: ['NISAR_T162A_F007_20260613_20260625', 'NISAR_T061A_F007_20260618_20260630', 'NISAR_T162A_F006_20260613_20260625']
   NISAR_T061A_F007_20260618_20260630: solape 116,541 px · desfase -516 mm (solape) · RMS solape 147 mm
   NISAR_T162A_F006_20260613_20260625: solape 61,361 px · desfase +125 mm (solape) · RMS solape 171 mm

== NISAR DESC: ['NISAR_T126D_F083_20260622_20260704', 'NISAR_T054D_F083_20260617_20260629', 'NISAR_T126D_F084_20260622_20260704']
   NISAR_T054D_F083_20260617_20260629: solape 103,160 px · desfase +157 mm (solape) · RMS solape 44 mm

TypeError: float() argument must be a string or a real number, not 'NoneType'

### C2 · Métricas y gráficos comparables de toda la interferometría / Comparable metrics and plots for all InSAR
🇪🇸 Misma ventana, misma escala de color (± cm común) y mismo contexto para todos los productos (Sentinel-1 y NISAR, incluido el par pre-sísmico de control). Genera la tabla de métricas (CSV ES/EN), mapas regional y costa, y un perfil común.
🇬🇧 Same window, same colour scale (common ± cm) and same context for every product (Sentinel-1 and NISAR, including the pre-seismic control pair). Produces the metrics table (CSV ES/EN), regional and coastal maps, and a common profile.

In [ ]:
P("insar_metricas")   # opcional / optional: variables={"LIM_MM": 300, "COH_MIN": 0.3}

## D · Daño: Umbra X + Planet + referencia / *Damage*
🇪🇸 Las figuras por sitio del 19-nov **no son publicables** (jet, título interno, Esri): tras actualizar `umbra_planet_sitios.py` al estilo nuevo use `forzar=True`. 🇬🇧 The 19 Nov per-site figures are not publishable; re-run with `forzar=True` after the style update.

### D2 · Óptico de alta resolución antes/después (Vantor abierto) / High-resolution optical before/after (open Vantor)
🇪🇸 Busca escenas de ~0,4 m anteriores y posteriores al sismo sobre cada sitio (sin descargas completas) y guarda un CSV y recortes de prueba. Licencia CC BY-NC 4.0: no subir a GitHub.
🇬🇧 Finds ~0.4 m pre- and post-event scenes over each site (no full downloads) and saves a CSV and test chips. CC BY-NC 4.0: do not commit.

In [ ]:
P("vantor", variables={"SITIOS_VANTOR": {"Caracas_Este_Petare": (10.4835, -66.8070)},   # (lat, lon) ← cambiar por el centro real / replace with true centre
                                     "LADO_M": 1000})

In [ ]:
P("danio_ref")

In [ ]:
P("danio_sitios", variables={"LADO_M": 1000, "ZOOM_M": 400, "SITIOS_A_PROCESAR": ["Catia_La_Mar", "Playa_Grande_Urimare", "La_Guaira"]})

## E · Figuras y tablas finales / *Final figures and tables*
🇪🇸 **Texto principal (≤ 10 figuras, ≤ 3 tablas):** Fig. 1 ubicación y cobertura · Fig. 3–5 LOS y Este/Vertical C vs L · Fig. 6 coherencia por cobertura · Fig. 7 ventana postsísmica · Fig. 8 daño Umbra × Planet · Fig. 9 daño vs gradiente/distancia a la falla · Fig. 10 infraestructura · Tablas 1–3.
**Suplemento (sin límite):** S1 línea de tiempo · S2 huellas por sensor (Sentinel-1, NISAR, ALOS-2, Umbra, Planet) · Tabla S1 inventario completo · S3 los 6 sitios de daño · S4 paletas y pruebas de gris.
🇬🇧 Scripts for Figs. 6, 7, 9, 10 and Table 3 are **pending**; they will be added to `src/` and as steps of this notebook (`PASOS` in `src/maestro.py`).

In [ ]:
P("exportar")   # figuras a GitHub: NO subir imágenes Planet (CC BY-NC) ni Esri / do not upload Planet or Esri imagery

## Estado final / *Final status*

In [ ]:
maestro.estado(RAIZ)